# Train the plate finder (YOLO11n) on Colab

Run the cells **top to bottom**. Before you start: **Runtime > Change runtime type > T4 GPU**.

Needs `yolo_for_training.zip` at the top level of your Google Drive ("My Drive").
Trained weights are saved to `MyDrive/runs/plates/weights/best.pt`, so they survive a disconnect.

## 1. Check the GPU

In [ ]:
!nvidia-smi
# You should see a table with "Tesla T4" (or similar). If it says "command not found",
# the runtime has no GPU: Runtime > Change runtime type > T4 GPU, then run this cell again.

## 2. Connect Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Check the upload is complete
The size should be **795769047** bytes, and the second line should say *No errors detected*.

In [ ]:
ZIP = '/content/drive/MyDrive/yolo_for_training.zip'
!ls -l "{ZIP}"
!unzip -tq "{ZIP}"

## 4. Unpack onto Colab's fast local disk (training from Drive directly is very slow)

In [ ]:
!unzip -q "{ZIP}" -d /content
print('train images:', len(__import__('os').listdir('/content/yolo/images/train')), '(expect 8299)')
print('val images:  ', len(__import__('os').listdir('/content/yolo/images/val')),   '(expect 1127)')
print('test images: ', len(__import__('os').listdir('/content/yolo/images/test')),  '(expect 1020)')
!cat /content/yolo/plates_colab.yaml

## 5. Install YOLO

In [ ]:
!pip install -q ultralytics
import ultralytics
ultralytics.checks()

## 6. Quick test (about 2 minutes)
Trains for 1 epoch on 2% of the data. The accuracy will be near zero: this only checks that nothing is broken.

In [ ]:
from ultralytics import YOLO

DATA = '/content/yolo/plates_colab.yaml'

smoke = YOLO('yolo11n.pt')          # downloads the pretrained starting model
smoke.train(data=DATA, epochs=1, fraction=0.02, imgsz=640, batch=16,
            device=0, project='/content/runs', name='smoke', exist_ok=True, plots=False)

## 7. Real training
`epochs=50` is a guess for a T4: expect roughly 1 to 2 minutes per epoch, so about 1 to 1.5 hours (a rough estimate).
`patience=15` stops early if the validation score has not improved for 15 epochs.
If you get an *out of memory* error, change `batch=32` to `batch=16`.

Re-running this cell **starts over and overwrites** the previous run. To continue after a disconnect, use cell 7b instead.

In [ ]:
RUN = '/content/drive/MyDrive/runs/plates'      # saved to Drive so a disconnect cannot lose it

model = YOLO('yolo11n.pt')
model.train(
    data=DATA,
    epochs=50,
    patience=15,
    imgsz=640,
    batch=32,
    device=0,
    workers=2,
    project='/content/drive/MyDrive/runs',
    name='plates',
    exist_ok=True,
)

### 7b. Only if Colab disconnected mid-training
Re-run cells 1 to 5, then run this. It continues from the last saved epoch.

In [ ]:
# RUN = '/content/drive/MyDrive/runs/plates'
# YOLO(f'{RUN}/weights/last.pt').train(resume=True)

## 8. Look at the training curves
Good signs: both losses go **down**, and `mAP50` goes **up** and flattens.
Warning sign: validation loss rising while training loss keeps falling means the model is memorizing.

In [ ]:
from IPython.display import Image, display
RUN = '/content/drive/MyDrive/runs/plates'
display(Image(f'{RUN}/results.png', width=900))

## 9. Test on images the model has never seen
Reminder: the test split is a little optimistic, because near-copies of some images can sit in both train and test.

In [ ]:
best = YOLO(f'{RUN}/weights/best.pt')
m = best.val(data=DATA, split='test', imgsz=640, device=0)
print(f'precision  {m.box.mp:.3f}   (of boxes drawn, share that were real plates)')
print(f'recall     {m.box.mr:.3f}   (of real plates, share that were found)')
print(f'mAP50      {m.box.map50:.3f}   (overall score, 0 to 1)')

## 10. Recall by plate width (your research question)
The score above averages big and small plates together. This cell asks: **how often are small plates found?**
A real plate counts as "found" if some predicted box overlaps it with IoU of at least 0.5.

In [ ]:
import glob, os
import numpy as np

def iou(a, b):
    ix1, iy1, ix2, iy2 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    union = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / union if union > 0 else 0

BANDS = [('<40', 0, 40), ('40-80', 40, 80), ('80-150', 80, 150), ('150+', 150, 1e9)]
found = {b[0]: 0 for b in BANDS}
total = {b[0]: 0 for b in BANDS}

images = sorted(glob.glob('/content/yolo/images/test/*'))
for r in best.predict(source=images, conf=0.25, imgsz=640, device=0, stream=True, verbose=False):
    h, w = r.orig_shape                                   # original picture size in pixels
    stem = os.path.splitext(os.path.basename(r.path))[0]
    preds = r.boxes.xyxy.cpu().numpy()                    # predicted boxes, in pixels
    for line in open(f'/content/yolo/labels/test/{stem}.txt').read().splitlines():
        p = line.split()
        if len(p) != 5:
            continue
        cx, cy, bw, bh = (float(v) for v in p[1:])
        gt = [(cx-bw/2)*w, (cy-bh/2)*h, (cx+bw/2)*w, (cy+bh/2)*h]
        width = gt[2] - gt[0]
        band = next(n for n, lo, hi in BANDS if lo <= width < hi)
        total[band] += 1
        found[band] += any(iou(gt, pb) >= 0.5 for pb in preds)

for name, _, _ in BANDS:
    n = total[name]
    print(f'{name:>7} px   plates: {n:<5}  found: {100*found[name]/max(n,1):5.1f}%')

## 11. See the predictions (look at these, numbers can hide problems)

In [ ]:
import random
from PIL import Image as PILImage
import matplotlib.pyplot as plt

random.seed(1)
sample = random.sample(images, 8)
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, r in zip(axes.flat, best.predict(source=sample, conf=0.25, imgsz=640, device=0, verbose=False)):
    ax.imshow(r.plot()[..., ::-1])       # r.plot() draws the boxes; [..., ::-1] fixes colour order
    ax.axis('off')
plt.tight_layout()
plt.show()

## 12. Export a fast version and download the weights

In [ ]:
onnx_path = best.export(format='onnx', imgsz=640)      # runs on almost anything
print('exported:', onnx_path)

from google.colab import files
files.download(f'{RUN}/weights/best.pt')               # the main file you need